In [ ]:
import nltk
nltk.download('averaged_perceptron_tagger_eng')

In [ ]:
import pandas as pd
import re
import numpy as np
import torch
from torch.utils.data import Dataset, DataLoader
from transformers import BertTokenizer, BertForSequenceClassification
from torch.optim import AdamW
from torch.optim.lr_scheduler import CosineAnnealingWarmRestarts
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score, classification_report
from sklearn.model_selection import StratifiedKFold
from nltk.tokenize import TweetTokenizer
from nltk.stem import WordNetLemmatizer
from nltk.corpus import wordnet
from nltk import pos_tag
from nltk.corpus import stopwords
from emoji import demojize
from tqdm import tqdm
import matplotlib.pyplot as plt
from torch.optim.lr_scheduler import ReduceLROnPlateau

In [ ]:
ABBREVIATIONS = {
    "he's": "he is", "there's": "there is", "We're": "We are", "That's": "That is",
    "won't": "will not", "they're": "they are", "Can't": "Can not", "wasn't": "was not",
    "don't": "do not", "aren't": "are not", "isn't": "is not", "What's": "What is",
    "haven't": "have not", "hasn't": "has not", "There's": "There is", "He's": "He is",
    "It's": "It is", "You're": "You are", "I'M": "I am", "shouldn't": "should not",
    "wouldn't": "would not", "i'm": "I am", "I'm": "I am", "Isn't": "is not",
    "Here's": "Here is", "you've": "you have", "we're": "we are", "what's": "what is",
    "couldn't": "could not", "we've": "we have", "it's": "it is", "doesn't": "does not",
    "It's": "It is", "who's": "who is", "I've": "I have", "y'all": "you all",
    "would've": "would have", "it'll": "it will", "we'll": "we will", "We've": "We have",
    "he'll": "he will", "Y'all": "You all", "Weren't": "Were not", "Didn't": "Did not",
    "they'll": "they will", "they'd": "they would", "DON'T": "DO NOT", "That's": "That is",
    "they've": "they have", "i'd": "I would", "should've": "should have", "You're": "You are",
    "where's": "where is", "we'd": "we would", "i'll": "I will", "weren't": "were not",
    "They're": "They are", "you'll": "you will", "I'd": "I would", "let's": "let us",
    "it's": "it is", "can't": "can not", "don't": "do not", "you're": "you are",
    "i've": "I have", "that's": "that is", "i'll": "I will", "doesn't": "does not",
    "i'd": "I would", "didn't": "did not", "ain't": "am not", "I've": "I have",
    "Don't": "do not", "I'll": "I will", "I'd": "I would", "Let's": "Let us",
    "you'd": "You would", "It's": "It is", "Ain't": "am not", "Haven't": "Have not",
    "Could've": "Could have", "youve": "you have", "w/": "with", "w/o": "without",
    "b/c": "because", "thx": "thanks", "u": "you", "r": "are", "rt": "retweet"
}

In [ ]:
class CustomTweetCleaner:
    def __init__(self):
        self.tokenizer = TweetTokenizer(preserve_case=False, reduce_len=True, strip_handles=True)
        self.lemmatizer = WordNetLemmatizer()

    def get_wordnet_pos(self, treebank_tag):
        """将 NLTK 的 POS 标签转换为 WordNet 的 POS 标签"""
        if treebank_tag.startswith('J'):
            return wordnet.ADJ
        elif treebank_tag.startswith('V'):
            return wordnet.VERB
        elif treebank_tag.startswith('N'):
            return wordnet.NOUN
        elif treebank_tag.startswith('R'):
            return wordnet.ADV
        else:
            return wordnet.NOUN

    def clean_text(self, text):
        """清洗文本数据"""
        stop_words = set(stopwords.words('english'))
        text = demojize(text)
        text = re.sub(r'#(\w+)', r'\1', text)
        text = re.sub(r'http[s]?://(?:[a-zA-Z]|[0-9]|[$-_@.&+]|[!*\\(\\),]|(?:%[0-9a-fA-F][0-9a-fA-F]))+', '', text)
        for abbr, full in ABBREVIATIONS.items():
            text = re.sub(r'\b' + re.escape(abbr) + r'\b', full, text, flags=re.IGNORECASE)
        text = re.sub(r'[^\w\s]', '', text)
        tokens = self.tokenizer.tokenize(text)
        tagged_tokens = pos_tag(tokens)
        lemmatized_tokens = [
            self.lemmatizer.lemmatize(token, self.get_wordnet_pos(pos))
            for token, pos in tagged_tokens
            if token.lower() not in stop_words
        ]
        return ' '.join(lemmatized_tokens).strip()

    def tokenize(self, text):
        """自定义分词器"""
        cleaned_text = self.clean_text(text)
        tokens = self.tokenizer.tokenize(cleaned_text)
        return tokens

In [ ]:
! pip install nlpaug

In [ ]:
import pandas as pd
import nlpaug.augmenter.word as naw



def augment_data(data, target_size=10000):
    # 初始化数据增强器
    aug = naw.SynonymAug(aug_src='wordnet', aug_p=0.3)
    
    # 当前数据量和需要增加的数量
    current_size = len(data)
    samples_needed = target_size - current_size

    if samples_needed <= 0:
        print("No augmentation needed.")
        return data

    # 类别分布
    class_counts = data['target'].value_counts()
    minority_class = 1 if class_counts.get(1, 0) < class_counts.get(0, 0) else 0
    minority_samples = int(samples_needed * 0.7)
    majority_samples = samples_needed - minority_samples

    augmented_texts = []
    augmented_targets = []

    # 增强少数类
    minority_data = data[data['target'] == minority_class]
    for _ in range(minority_samples):
        text = minority_data['text'].sample(1).iloc[0]
        aug_text = aug.augment(text)[0]
        augmented_texts.append(aug_text)
        augmented_targets.append(minority_class)

    # 增强多数类
    majority_data = data[data['target'] != minority_class]
    for _ in range(majority_samples):
        text = majority_data['text'].sample(1).iloc[0]
        aug_text = aug.augment(text)[0]
        augmented_texts.append(aug_text)
        augmented_targets.append(1 - minority_class)

    # 创建增强数据 DataFrame，保持原始列结构（除 text, target 外置为 None）
    empty_columns = {col: [None] * samples_needed for col in data.columns if col not in ['text', 'target']}
    augmented_df = pd.DataFrame({
        'text': augmented_texts,
        'target': augmented_targets,
        **empty_columns
    })

    # 拼接回原始数据
    final_data = pd.concat([data, augmented_df], ignore_index=True)

    # 打印结果
    print(f"Data size after augmentation: {len(final_data)}")
    print(f"Class distribution after augmentation:\n{final_data['target'].value_counts()}")

    return final_data

In [ ]:
class TweetDataset(Dataset):
    def __init__(self, texts, keywords, labels, tokenizer, max_len=128):
        self.texts = texts
        self.keywords = keywords
        self.labels = labels
        self.tokenizer = tokenizer
        self.max_len = max_len
        self.cleaner = CustomTweetCleaner()

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        text = self.texts[idx]
        keyword = self.keywords[idx] if self.keywords[idx] else 'none'
        label = self.labels[idx]

        cleaned_text = self.cleaner.clean_text(text)
        cleaned_keyword = self.cleaner.clean_text(keyword)
        input_text = f"{cleaned_keyword} [SEP] {cleaned_text}"

        encoding = self.tokenizer.encode_plus(
            input_text,
            add_special_tokens=True,
            max_length=self.max_len,
            padding='max_length',
            truncation=True,
            return_attention_mask=True,
            return_tensors='pt'
        )

        return {
            'input_ids': encoding['input_ids'].flatten(),
            'attention_mask': encoding['attention_mask'].flatten(),
            'labels': torch.tensor(label, dtype=torch.long)
        }

In [ ]:
class DisasterDetector:
    def __init__(self, bert_model_path, max_seq_length=128, lr=2e-5, epochs=6, batch_size=32, n_splits=3):
        self.bert_model_path = bert_model_path
        self.max_seq_length = max_seq_length
        self.lr = lr
        self.epochs = epochs
        self.batch_size = batch_size
        self.n_splits = n_splits
        self.tokenizer = None
        self.models = []  # 存储每个Fold的模型
        self.scores = {}  # 存储每个Fold的评估Parameter
        self.device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

    def build_model(self):
        """构建 BERT 分类模型"""
        model = BertForSequenceClassification.from_pretrained(
            self.bert_model_path,
            num_labels=2,
            hidden_dropout_prob=0.2
        )
        model.to(self.device)
        return model

    def compute_class_weights(self, data):
        """计算类权重以处理不平衡数据"""
        class_counts = data['target'].value_counts()
        total_samples = len(data)
        class_weights = {target: total_samples / (2 * class_counts[target]) for target in class_counts.index}
        weights = [class_weights[0], class_weights[1]]
        return torch.tensor(weights, dtype=torch.float).to(self.device)

    def train(self, data):
        """训练模型，使用 K FoldFind """
        self.tokenizer = BertTokenizer.from_pretrained(self.bert_model_path)
        skf = StratifiedKFold(n_splits=self.n_splits, shuffle=True, random_state=42)
        X = data['text'].values
        X_keyword = data['keyword'].values
        y = data['target'].values

        class_weights = self.compute_class_weights(data)
        print(f"类权重: {class_weights}")

        for fold, (trn_idx, val_idx) in enumerate(skf.split(X, y), 1):

            X_train = X[trn_idx].tolist()
            X_val = X[val_idx].tolist()
            X_train_keyword = X_keyword[trn_idx].tolist()
            X_val_keyword = X_keyword[val_idx].tolist()
            y_train = y[trn_idx].tolist()
            y_val = y[val_idx].tolist()

            train_dataset = TweetDataset(X_train, X_train_keyword, y_train, self.tokenizer, self.max_seq_length)
            val_dataset = TweetDataset(X_val, X_val_keyword, y_val, self.tokenizer, self.max_seq_length)
            train_loader = DataLoader(train_dataset, batch_size=self.batch_size, shuffle=True)
            val_loader = DataLoader(val_dataset, batch_size=self.batch_size)

            model = self.build_model()
            optimizer = AdamW(model.parameters(), lr=self.lr, weight_decay=0.05)
            scheduler = CosineAnnealingWarmRestarts(optimizer, T_0=2, T_mult=2, eta_min=1e-6)
            #scheduler = ReduceLROnPlateau(optimizer, mode='min', factor=0.8, patience=1, verbose=True)
            loss_fn = torch.nn.CrossEntropyLoss(weight=class_weights, label_smoothing=0.1)

            best_f1 = 0
            patience = 2
            early_stop_counter = 0

            fold_train_losses = []
            fold_val_losses = []
            fold_val_accuracies = []
            fold_val_f1_scores = []
            fold_val_precisions = []
            fold_val_recalls = []

            for epoch in range(self.epochs):
                model.train()
                total_train_loss = 0
                for batch in tqdm(train_loader, desc=f"Fold {fold} Epoch {epoch + 1} 训练"):
                    input_ids = batch['input_ids'].to(self.device)
                    attention_mask = batch['attention_mask'].to(self.device)
                    labels = batch['labels'].to(self.device)
                    optimizer.zero_grad()
                    outputs = model(input_ids, attention_mask=attention_mask)
                    logits = outputs.logits
                    loss = loss_fn(logits, labels)
                    total_train_loss += loss.item()
                    loss.backward()
                    torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
                    optimizer.step()
                    scheduler.step()

                avg_train_loss = total_train_loss / len(train_loader)
                fold_train_losses.append(avg_train_loss)

                model.eval()
                total_val_loss = 0
                predictions = []
                true_labels = []
                with torch.no_grad():
                    for batch in tqdm(val_loader, desc=f"Fold {fold} Epoch {epoch + 1} 验证"):
                        input_ids = batch['input_ids'].to(self.device)
                        attention_mask = batch['attention_mask'].to(self.device)
                        labels = batch['labels'].to(self.device)
                        outputs = model(input_ids, attention_mask=attention_mask)
                        logits = outputs.logits
                        loss = loss_fn(logits, labels)
                        total_val_loss += loss.item()
                        preds = torch.argmax(logits, dim=1)
                        predictions.extend(preds.cpu().numpy())
                        true_labels.extend(labels.cpu().numpy())

                avg_val_loss = total_val_loss / len(val_loader)
                val_accuracy = accuracy_score(true_labels, predictions)
                val_f1 = f1_score(true_labels, predictions, average='weighted')
                val_precision = precision_score(true_labels, predictions, average='weighted')
                val_recall = recall_score(true_labels, predictions, average='weighted')

                fold_val_losses.append(avg_val_loss)
                fold_val_accuracies.append(val_accuracy)
                fold_val_f1_scores.append(val_f1)
                fold_val_precisions.append(val_precision)
                fold_val_recalls.append(val_recall)

                print(f"Fold {fold} Epoch {epoch + 1}:")
                print(f"  训练Loss: {avg_train_loss:.4f}")
                print(f"  验证Loss: {avg_val_loss:.4f}")
                print(f"  验证准确率: {val_accuracy:.4f}")
                print(f"  验证 F1 分数: {val_f1:.4f}")
                print(f"  验证精确率: {val_precision:.4f}")
                print(f"  验证召回率: {val_recall:.4f}")


                if val_f1 > best_f1:
                    best_f1 = val_f1
                    early_stop_counter = 0
                else:
                    early_stop_counter += 1
                    if early_stop_counter >= patience:
                        print(f"早停触发于Epoch {epoch + 1}")
                        break

            self.models.append(model)
            self.scores[fold] = {
                'train': {
                    'loss': fold_train_losses,
                },
                'validation': {
                    'loss': fold_val_losses,
                    'accuracy': fold_val_accuracies,
                    'f1': fold_val_f1_scores,
                    'precision': fold_val_precisions,
                    'recall': fold_val_recalls
                }
            }

            self.plot_classification_metrics(len(fold_train_losses), fold_val_accuracies, fold_val_f1_scores,
                                            fold_val_precisions, fold_val_recalls, self.batch_size, self.lr, fold)
            self.plot_loss_curves(len(fold_train_losses), fold_train_losses, fold_val_losses, self.batch_size, self.lr, fold)

            print(f"\nFold {fold} 最终评估:")
            print(f"准确率: {val_accuracy:.4f}")
            print(f"F1 分数: {val_f1:.4f}")
            print(f"精确率: {val_precision:.4f}")
            print(f"召回率: {val_recall:.4f}")
            print("\n分类报告:\n", classification_report(true_labels, predictions))

    def plot_classification_metrics(self, epochs, val_accuracies, val_f1_scores, val_precisions, val_recalls, batch_size, learning_rate, fold=None):
        """绘制分类报告曲线：准确率、F1 分数、精确率、召回率"""
        plt.figure(figsize=(12, 8))  # 增大图表尺寸以铺满画面
        plt.plot(range(1, epochs + 1), val_accuracies, label='Val Accuracy', color='blue', marker='o')
        plt.plot(range(1, epochs + 1), val_f1_scores, label='Val F1 Score', color='gold', marker='o')
        plt.plot(range(1, epochs + 1), val_precisions, label='Val Precision', color='cyan', marker='o')
        plt.plot(range(1, epochs + 1), val_recalls, label='Val Recall', color='purple', marker='o')
        plt.xlabel('Epoch')
        plt.ylabel('Metric Value')

        # 动态计算 y 轴范围以放大波动
        all_metrics = np.concatenate([val_accuracies, val_f1_scores, val_precisions, val_recalls])
        y_min, y_max = np.min(all_metrics), np.max(all_metrics)
        margin = (y_max - y_min) * 0.1  # 添加 10% 的边距
        plt.ylim(max(0, y_min - margin), min(1, y_max + margin))  # 限制在 [0, 1] 范围内

        lr_str = f"{learning_rate:.0e}"
        if fold is not None:
            plt.title(f'Fold {fold} Parameter (Batch Size={batch_size}, Learning Rate={lr_str})')
        else:
            plt.title(f'Find Parameter (Batch Size={batch_size}, Learning Rate={lr_str})')
        plt.legend()
        plt.grid(True, which='both', linestyle='--', linewidth=0.5)  # 添加更细的网格线
        plt.xticks(range(1, epochs + 1))
        plt.minorticks_on()  # 启用次刻度线以便更细致观察
        plt.tight_layout()  # 优化布局，防止标签被裁剪
        plt.show()
        plt.close()

    def plot_loss_curves(self, epochs, train_losses, val_losses, batch_size, learning_rate, fold=None):
        """绘制Loss曲线：训练Loss、验证Loss"""
        plt.figure(figsize=(12, 8))  # 增大图表尺寸以铺满画面
        plt.plot(range(1, epochs + 1), train_losses, label='Train Loss', color='cyan', marker='o')
        plt.plot(range(1, epochs + 1), val_losses, label='Val Loss', color='red', marker='o')
        plt.xlabel('Epoch')
        plt.ylabel('Loss')

        # 动态计算 y 轴范围以放大波动
        all_losses = np.concatenate([train_losses, val_losses])
        y_min, y_max = np.min(all_losses), np.max(all_losses)
        margin = (y_max - y_min) * 0.1  # 添加 10% 的边距
        plt.ylim(max(0, y_min - margin), y_max + margin)  # 动态设置 y 轴范围

        lr_str = f"{learning_rate:.0e}"
        if fold is not None:
            plt.title(f'Fold {fold} Loss (Batch Size={batch_size}, Learning Rate={lr_str})')
        else:
            plt.title(f'Find Loss (Batch Size={batch_size}, Learning Rate={lr_str})')
        plt.legend()
        plt.grid(True, which='both', linestyle='--', linewidth=0.5)  # 添加更细的网格线
        plt.xticks(range(1, epochs + 1))
        plt.minorticks_on()  # 启用次刻度线以便更细致观察
        plt.tight_layout()  # 优化布局，防止标签被裁剪
        plt.show()
        plt.close()

   

    def predict(self, test_data, return_probs=False):
        if self.device.type == 'cuda' and not torch.cuda.is_available():
            print("警告: GPU 不可用，切换到 CPU")
            self.device = torch.device('cpu')
            for model in self.models:
                model.to(self.device)

        cleaner = CustomTweetCleaner()
        test_data['text_cleaned'] = test_data['text'].apply(cleaner.clean_text)
        test_data['keyword'] = test_data['keyword'].fillna('none').apply(cleaner.clean_text)

        test_dataset = TweetDataset(
            test_data['text'].tolist(),
            test_data['keyword'].tolist(),
            [0] * len(test_data),  # 占位标签
            self.tokenizer,
            self.max_seq_length
        )
        test_loader = DataLoader(test_dataset, batch_size=self.batch_size, shuffle=False)

        predictions = []
        probabilities = []
        for model in self.models:
            model.eval()
            model_preds = []
            model_probs = []
            with torch.no_grad():
                for batch in tqdm(test_loader, desc="预测测试集"):
                    input_ids = batch['input_ids'].to(self.device)
                    attention_mask = batch['attention_mask'].to(self.device)
                    outputs = model(input_ids, attention_mask=attention_mask)
                    probs = torch.softmax(outputs.logits, dim=1)[:, 1]  # 正类概率
                    preds = torch.argmax(outputs.logits, dim=1)  # 预测类别
                    model_preds.extend(preds.cpu().numpy())
                    model_probs.extend(probs.cpu().numpy())
            predictions.append(model_preds)
            probabilities.append(model_probs)
            torch.cuda.empty_cache()  # 清理 GPU 内存

        predictions = np.array(predictions).T  # 形状: (n_samples, n_models)
        final_preds = np.apply_along_axis(lambda x: np.bincount(x).argmax(), axis=1, arr=predictions)
        final_probs = np.mean(probabilities, axis=0) if return_probs else None

        submission = pd.DataFrame({'id': test_data['id'], 'target': final_preds})
        if return_probs:
            submission['probability'] = final_probs
        return submission

In [ ]:
data = pd.read_csv('/kaggle/input/nlp-getting-started/train.csv')
test_data = pd.read_csv('/kaggle/input/nlp-getting-started/test.csv')

# 数据清洗
cleaner = CustomTweetCleaner()
data['text'] = data['text'].apply(cleaner.clean_text)
data=augment_data(data)
data['keyword'] = data['keyword'].fillna('none').apply(cleaner.clean_text)


In [ ]:
# 初始化并训练模型
bert_model_path = r"/kaggle/input/bert_model/transformers/default/1/bert_model"
detector = DisasterDetector(bert_model_path, max_seq_length=128, lr=2e-5, epochs=4, batch_size=32, n_splits=3)
detector.train(data)

In [ ]:
# 预测测试集
submission = detector.predict(test_data)

In [ ]:
submission.to_csv("submission.csv", index=False)

In [ ]:
submission.head()